In [1]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString
from pathlib import Path
import torch

In [2]:
traj_cleaned_df = pd.read_parquet("../data/nyc_output_tabular/output/traj_cleaned.parquet")
traj_cleaned_df.head()

,datetime,uid,tid,lat,lng
0,2021-05-23 08:23:21,3,3,40.821955,-74.139058
1,2021-05-23 08:23:24,3,3,40.822035,-74.139246
2,2021-05-23 08:23:26,3,3,40.822115,-74.139433
3,2021-05-23 08:23:28,3,3,40.822036,-74.139485
4,2021-05-23 08:23:32,3,3,40.821987,-74.139464


In [3]:
traj_cleaned_df["datetime"].head()

0   2021-05-23 08:23:21
1   2021-05-23 08:23:24
2   2021-05-23 08:23:26
3   2021-05-23 08:23:28
4   2021-05-23 08:23:32
Name: datetime, dtype: datetime64[ns]

In [10]:
traj_cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6625802 entries, 0 to 6625801
Data columns (total 5 columns):
 #   Column    Dtype         
---  ------    -----         
 0   datetime  datetime64[ns]
 1   uid       int64         
 2   tid       str           
 3   lat       float64       
 4   lng       float64       
dtypes: datetime64[ns](1), float64(2), int64(1), str(1)
memory usage: 285.6 MB


## Build an FMM-ready edge file from OSMnx

creates a directed edge layer with columns fid, u, v, and geometry, which matches the common FMM pattern

`uv pip install osmnx`

In [ ]:
EDGES_GDF_OUTPUT_PATH = "../outputs/nyc_output_tabular/nyc_edges.gpkg"
output_path = Path(EDGES_GDF_OUTPUT_PATH)
# {"all", "all_public", "bike", "drive", "drive_service", "walk"} What type of street network to retrieve
NETWORK_TYPE = "drive"

In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def build_fmm_network_from_place(place: str, out_shp: str) -> None:
    """ Build the road network with OSMnx.

        the exported network needs fid, u, and v columns to be compatible with FMM
    """
    # output_dir_path = Path(out_dir)
    # output_dir_path.parent.mkdir(parents=True, exist_ok=True)

    # Download drivable network
    G = ox.graph_from_place(place, network_type="drive")

    # Convert to GeoDataFrames
    nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

    # Flatten (u, v, key) index into columns
    edges_gdf = edges_gdf.reset_index()

    # FMM-compatible columns
    edges_gdf["fid"] = np.arange(len(edges_gdf), dtype="int64")

    # Keep required columns first
    keep = ["fid", "u", "v", "geometry"]
    optional = ["osmid", "length", "highway", "name", "oneway"]
    for col in optional:
        if col not in edges_gdf.columns:
            edges_gdf[col] = None

    edges_gdf = edges_gdf[keep + optional].copy()

    # shp_path = os.path.join(out_dir, "edges.shp")
    # shp_path = output_dir_path / "edges.shp"
    edges_gdf.to_file(out_shp)

In [ ]:
# Pick one:
# Better if you want tighter control:
# north, south, east, west = 40.93, 40.47, -73.68, -74.30
# G = ox.graph_from_bbox((north, south, east, west), network_type="drive")
G = ox.graph_from_place("New York City, New York, USA", network_type=NETWORK_TYPE)

In [20]:
# Convert to GeoDataFrames
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

# Flatten the multi-index (u, v, key)
edges_gdf = edges_gdf.reset_index()

In [ ]:
# Keep only what we need first
keep_cols = ["u", "v", "geometry"]
# maxspeed in mph
optional_cols = ["osmid", "length", "highway", "name", "oneway"]
for c in optional_cols:
    if c not in edges_gdf.columns:
        edges_gdf[c] = None

edges_gdf = edges_gdf[keep_cols + optional_cols].copy()

maxspeed column has about 97039 null values so not using it as about 70% dont have a max speed.

In [26]:
# preprocessing
# remove white space and 'mph' in maxspeed
# edges_gdf["maxspeed"] = edges_gdf["maxspeed"].str.replace(r"\s+|mph", "", regex=True)

In [27]:
# edges_gdf["maxspeed"].value_counts()

In [ ]:
# edges_gdf["maxspeed"].isnull().sum()

np.int64(97039)

In [ ]:
# edges_gdf["maxspeed"].shape

(139156,)

In [28]:
edges_gdf.head(3)

,u,v,geometry,osmid,length,highway,name,oneway,maxspeed
0,39076461,274283981,"LINESTRING (-73.79475 40.78635, -73.79462 40.7...",25161349,819.501666,motorway,Cross Island Parkway,True,50 mph
1,39076461,42854803,"LINESTRING (-73.79475 40.78635, -73.79332 40.7...",25161578,268.144095,motorway_link,NaN,True,NaN
2,39076490,277672046,"LINESTRING (-73.75709 40.76243, -73.75721 40.7...",5699971,246.749804,motorway_link,NaN,True,NaN


In [ ]:
# Create a unique edge id for FMM
edges_gdf["fid"] = range(len(edges_gdf))

# Reorder columns
edges_gdf = edges_gdf[
    ["fid", "u", "v", "geometry", "osmid", "length", "highway", "name", "oneway"]
]

# Save as GeoPackage (preferred) or shapefile
edges_gdf.to_file(EDGES_GDF_OUTPUT_PATH, layer="edges", driver="GPKG")
# or:
# edges_gdf.to_file("nyc_edges.shp")

In [4]:
for u, v, k, data in G.edges(data=True, keys=True):
    print(f"Edge ({u}, {v}) with key {k} has attributes: {data}")
    break

Edge (39076461, 274283981) with key 0 has attributes: {'osmid': 25161349, 'highway': 'motorway', 'lanes': '2', 'maxspeed': '50 mph', 'name': 'Cross Island Parkway', 'oneway': True, 'ref': 'CI', 'reversed': False, 'length': np.float64(819.5016661477804), 'geometry': <LINESTRING (-73.795 40.786, -73.795 40.786, -73.794 40.786, -73.794 40.786,...>}


In [38]:
edges_loaded = gpd.read_file("../fmm_scripts/data/fmm_nyc.shp")

In [39]:
edges_loaded.head()

,fid,u,v,osmid,length,highway,name,oneway,geometry
0,0,39076461,274283981,25161349,819.501666,motorway,Cross Island Parkway,True,"LINESTRING (-73.79475 40.78635, -73.79462 40.7..."
1,1,39076461,42854803,25161578,268.144095,motorway_link,NaN,True,"LINESTRING (-73.79475 40.78635, -73.79332 40.7..."
2,2,39076490,277672046,5699971,246.749804,motorway_link,NaN,True,"LINESTRING (-73.75709 40.76243, -73.75721 40.7..."
3,3,39076490,277672005,1014007069,291.838695,motorway,Cross Island Parkway,True,"LINESTRING (-73.75709 40.76243, -73.75741 40.7..."
4,4,39076504,462124701,"[618709517, 618709515, 5700693]",433.149850,motorway_link,NaN,True,"LINESTRING (-73.74416 40.75347, -73.74453 40.7..."


In [4]:
edges_loaded.describe()

,fid,u,v,length
count,139155.000000,1.391550e+05,1.391550e+05,139155.000000
mean,69577.000000,9.655134e+08,9.645541e+08,115.627430
std,40170.732692,2.400862e+09,2.401345e+09,108.280257
min,0.000000,3.907646e+07,3.907646e+07,0.616284
25%,34788.500000,4.273538e+07,4.273533e+07,71.773251
50%,69577.000000,4.285637e+07,4.285618e+07,82.124825
75%,104365.500000,4.299150e+07,4.299090e+07,146.984278
max,139154.000000,1.370054e+10,1.370054e+10,3607.274943


In [5]:
edges_loaded.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 139155 entries, 0 to 139154
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   fid       139155 non-null  int64   
 1   u         139155 non-null  int64   
 2   v         139155 non-null  int64   
 3   osmid     139155 non-null  str     
 4   length    139155 non-null  float64 
 5   highway   139155 non-null  str     
 6   name      135594 non-null  str     
 7   oneway    139155 non-null  bool    
 8   geometry  139155 non-null  geometry
dtypes: bool(1), float64(1), geometry(1), int64(3), str(3)
memory usage: 13.2 MB


In [7]:
edges_loaded.isna().sum()

fid            0
u              0
v              0
osmid          0
length         0
highway        0
name        3561
oneway         0
geometry       0
dtype: int64

## Convert cleaned parquet to FMM GPS CSV

FMM accepts a CSV point file where each row is one observation with trajectory id, longitude, latitude, and optional timestamp; the file must already be sorted by id and timestamp

CSV point file: a CSV file with a header row and columns separated by ;. Each row stores a single observation containing id(integer), x(longitude), y(latitude), timestamp(optional, integer). The file must be sorted already by id and timestamp (trajectory will be passed sequentially). The id, x, y and timestamp column names will be specified by the user.

In [13]:
GPS_TRAJ_PARQUET_PATH = "../data/nyc_output_tabular/output/traj_cleaned.parquet"

GPS_TRAJ_FMM_PATH = "../outputs/nyc_output_tabular/nyc_gps_points_fmm_ready.csv"
output_path = Path(GPS_TRAJ_FMM_PATH)

In [14]:
output_path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def build_fmm_points_csv(
    parquet_path: str,
    out_csv: str,
    trip_id_map_csv: str | None = None
) -> pd.DataFrame:
    """
    CSV point file: a CSV file with a header row and columns separated by ;. 
    Each row stores a single observation containing id(integer), x(longitude), y(latitude), timestamp(optional, integer). 
    
    The file must be sorted already by id and timestamp (trajectory will be passed sequentially). The id, x, y and timestamp column names will be specified by the user.

    Since our tid is only unique within a user, build a unique trip key from uid + tid. FMM's docs say id is an integer, so remap each unique trip key to an integer trajectory id.
    """
    df = pd.read_parquet(parquet_path).copy()

    # Unique trip key
    df["trip_key"] = df["uid"].astype(str) + "_" + df["tid"].astype(str)

    # Integer trajectory ids for FMM
    trip_keys = pd.Index(df["trip_key"].unique())
    trip_id_map = pd.DataFrame({
        "trip_key": trip_keys,
        "id": range(len(trip_keys))
    })

    df = df.merge(trip_id_map, on="trip_key", how="left")

    # Integer unix timestamp in seconds
    df["timestamp"] = pd.to_datetime(df["datetime"]).astype("int64") // 10**9

    # Sort exactly as required
    df = df.sort_values(["id", "timestamp"]).reset_index(drop=True)

    gps_df = pd.DataFrame({
        "id": df["id"].astype("int64"),
        "x": df["lng"].astype(float),
        "y": df["lat"].astype(float),
        "timestamp": df["timestamp"].astype("int64"),
    })

    gps_df.to_csv(out_csv, sep=";", index=False)

    if trip_id_map_csv is not None:
        trip_id_map.to_csv(trip_id_map_csv, index=False)

    return trip_id_map

`uv pip install pyArrow`

In [15]:
df = pd.read_parquet(GPS_TRAJ_PARQUET_PATH).copy()

# Build one trajectory id per trip
df["trip_id"] = df["uid"].astype(str) + "_" + df["tid"].astype(str)

# Sort exactly as FMM expects
df = df.sort_values(["trip_id", "datetime"]).reset_index(drop=True)

# FMM point CSV wants longitude=x, latitude=y
gps_df = pd.DataFrame({
    "id": df["trip_id"],
    "x": df["lng"],
    "y": df["lat"],
    "timestamp": pd.to_datetime(df["datetime"]).astype("int64") // 10**9
})

# IMPORTANT: FMM point CSV uses semicolon separator
gps_df.to_csv(GPS_TRAJ_FMM_PATH, sep=";", index=False)

In [22]:
gps_df.describe()

,x,y,timestamp
count,6.625802e+06,6.625802e+06,6.625802e+06
mean,-7.406700e+01,4.070139e+01,1.668448e+09
std,8.884941e-02,9.993298e-02,4.029098e+07
min,-7.425900e+01,4.047704e+01,1.203292e+09
25%,-7.414831e+01,4.061133e+01,1.660215e+09
50%,-7.405205e+01,4.069093e+01,1.672397e+09
75%,-7.399015e+01,4.077322e+01,1.684830e+09
max,-7.389900e+01,4.091800e+01,1.728208e+09


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6625802 entries, 0 to 6625801
Data columns (total 6 columns):
 #   Column    Dtype         
---  ------    -----         
 0   datetime  datetime64[ns]
 1   uid       int64         
 2   tid       str           
 3   lat       float64       
 4   lng       float64       
 5   trip_id   str           
dtypes: datetime64[ns](1), float64(2), int64(1), str(2)
memory usage: 409.7 MB


In [8]:
df.isnull().sum()

datetime    0
uid         0
tid         0
lat         0
lng         0
trip_id     0
dtype: int64

## Create road network and prepare trajectory to be used by FMM

In [ ]:
BASE_DATA_DIR = "../data/nyc_output_tabular/output"
BASE_OUTPUT_DIR = "../outputs/nyc_output_tabular"

PLACE = "New York City, New York, USA"
ROAD_NETWORK_OUT_DIR = f"{BASE_OUTPUT_DIR}/fmm_nyc.shp"
road_network_out_dir_path = Path(ROAD_NETWORK_OUT_DIR)
road_network_out_dir_path.parent.mkdir(parents=True, exist_ok=True)

GPS_TRAJ_PARQUET_PATH = f"{BASE_DATA_DIR}/traj_cleaned.parquet"
GPS_TRAJ_FMM_OUTPUT_PATH = f"{BASE_OUTPUT_DIR}/nyc_gps_points_fmm_ready.csv"
TRIP_ID_MAP_CSV = f"{BASE_OUTPUT_DIR}/nyc_gps_points_fmm_trip_id_map.csv"
# {"all", "all_public", "bike", "drive", "drive_service", "walk"} What type of street network to retrieve
NETWORK_TYPE = "drive"
gps_traj_fmm_output_path = Path(GPS_TRAJ_FMM_OUTPUT_PATH)
gps_traj_fmm_output_path.parent.mkdir(parents=True, exist_ok=True)



build_fmm_network_from_place(PLACE, ROAD_NETWORK_OUT_DIR)

trip_map = build_fmm_points_csv(
    parquet_path=GPS_TRAJ_PARQUET_PATH,
    out_csv=GPS_TRAJ_FMM_OUTPUT_PATH,
    trip_id_map_csv=TRIP_ID_MAP_CSV,
)

## FMM Code

test text

```bash
sudo apt-get install -y \
    make \
    cmake \
    libboost-dev \
    libboost-serialization-dev \
    libgdal-dev \
    gdal-bin \
    libbz2-dev \
    libexpat1-dev \
    swig \
    python3-dev
```


It will build executable files under the build folder, which are installed to /usr/local/bin
```bash
git clone https://github.com/cyang-kth/fmm.git
cd fmm

mkdir build
cd build

cmake ..
make -j$(nproc)
sudo make install
```



### .geo file info

In [31]:
df_geo = pd.read_csv("../baselines/gan/ts_trajgen/outputs/nyc/nyc.geo")
df_geo.head(3)

,geo_id,type,coordinates
0,0,LineString,"[[-73.7947484,40.7863451],[-73.794615,40.78638..."
1,1,LineString,"[[-73.7947484,40.7863451],[-73.7933159,40.7867..."
2,2,LineString,"[[-73.7570906,40.7624294],[-73.7572099,40.7626..."


In [32]:
df_geo.info()

<class 'pandas.DataFrame'>
RangeIndex: 139155 entries, 0 to 139154
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   geo_id       139155 non-null  int64
 1   type         139155 non-null  str  
 2   coordinates  139155 non-null  str  
dtypes: int64(1), str(2)
memory usage: 20.9 MB


In [33]:
df_geo.describe()

,geo_id
count,139155.000000
mean,69577.000000
std,40170.732692
min,0.000000
25%,34788.500000
50%,69577.000000
75%,104365.500000
max,139154.000000


### .rel info

In [35]:
df_rel = pd.read_csv("../baselines/gan/ts_trajgen/outputs/nyc/nyc.rel")
df_rel.head()

,rel_id,type,origin_id,destination_id
0,0,geo,0,110446
1,1,geo,1,69063
2,2,geo,1,69064
3,3,geo,2,111196
4,4,geo,3,111193


In [36]:
df_rel.info()

<class 'pandas.DataFrame'>
RangeIndex: 384831 entries, 0 to 384830
Data columns (total 4 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   rel_id          384831 non-null  int64
 1   type            384831 non-null  str  
 2   origin_id       384831 non-null  int64
 3   destination_id  384831 non-null  int64
dtypes: int64(3), str(1)
memory usage: 12.8 MB


In [37]:
df_rel.describe()

,rel_id,origin_id,destination_id
count,384831.000000,384831.000000,384831.000000
mean,192415.000000,68921.121261,67794.107491
std,111091.285059,38940.296549,38492.057950
min,0.000000,0.000000,0.000000
25%,96207.500000,35937.500000,35305.000000
50%,192415.000000,68736.000000,67394.000000
75%,288622.500000,101121.000000,99132.000000
max,384830.000000,139154.000000,139154.000000


### nyc_fmm_match

In [5]:
df_mm_raw = pd.read_csv("../fmm_scripts/output/nyc_fmm_match.csv", sep=";")
df_mm_raw.head(3)

,id,opath,error,spdist,cpath,tpath,mgeom,length,duration,speed
0,0,NaN,NaN,NaN,NaN,NaN,LINESTRING(),NaN,"0,0,128,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",NaN
1,0,NaN,NaN,NaN,NaN,NaN,LINESTRING(),NaN,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",NaN
2,0,NaN,NaN,NaN,NaN,NaN,LINESTRING(),NaN,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",NaN


In [6]:
df_mm_raw.describe()

,id
count,18765.000000
mean,5243.868532
std,5676.156159
min,0.000000
25%,0.000000
50%,3631.000000
75%,9509.000000
max,18762.000000


In [7]:
df_mm_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 18765 entries, 0 to 18764
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        18765 non-null  int64
 1   opath     11199 non-null  str  
 2   error     11199 non-null  str  
 3   spdist    11116 non-null  str  
 4   cpath     11199 non-null  str  
 5   tpath     11116 non-null  str  
 6   mgeom     18765 non-null  str  
 7   length    11199 non-null  str  
 8   duration  18565 non-null  str  
 9   speed     11116 non-null  str  
dtypes: int64(1), str(9)
memory usage: 230.8 MB


In [10]:
df_mm_raw.isna().sum()

id             0
opath       7566
error       7566
spdist      7649
cpath       7566
tpath       7649
mgeom          0
length      7566
duration     200
speed       7649
dtype: int64

In [ ]:
print(f"{7566/18765 = }")

7566/18765 = 0.40319744204636293


about 40% of trajectories have no valid map-matching result

### Verifying nyc_mm_train/test.csv

In [12]:
def check_rid_and_timelist_alignment(mm_data: pd.DataFrame) -> bool:
    rid_lists = mm_data["rid_list"].str.split(",")
    time_lists = mm_data["time_list"].str.split(",")
    for rid_list, time_list in zip(rid_lists, time_lists):
        if len(rid_list) != len(time_list):
            print(f"rid_list found that does not have its complete time_list: {len(rid_list) = }, {len(time_list) = }")
            return False
    return True

In [13]:
def check_monotonic_time(mm_data: pd.DataFrame) -> bool:
    time_lists = mm_data["time_list"].str.split(",")
    for idx, time_list in enumerate(time_lists):
        if not all(time_list[i] <= time_list[i+1] for i in range(len(time_list)-1)):
            print(f"Found non-monotonic_times for a trajectory {time_list = }, {idx = }")
            return False
    return True

In [14]:
def check_rid_len_distribution(mm_data: pd.DataFrame) -> None:
    print(mm_data["rid_list"].apply(lambda x: len(x.split(","))).describe())
    # print(df["len"].describe())

In [15]:
# def time_list_interpolation_quality(mm_data: pd.DataFrame) -> None:
#     time_lists_str = mm_data["time_list"].str.split(",")
#     time_lists = pd.Series([pd.Series(pd.to_datetime(dt)) for dt in time_lists_str])
    
#     time_deltas = [time_list.diff() for time_list in time_lists]
#     print(pd.Series(time_deltas).describe())
#     # for time_list in time_lists:
#     #     time_deltas = time_list.diff()
def time_list_interpolation_quality(mm_data: pd.DataFrame) -> None:
    """
    computes per-trajectory deltas in SECONDS
    flattens everything for real statistics
    """
    all_deltas = []

    for time_str in mm_data["time_list"].dropna():
        # split + parse timestamps
        times = pd.to_datetime(time_str.split(","), errors="coerce")

        # compute deltas (in seconds)
        deltas = times.to_series().diff().dt.total_seconds()

        # drop NaNs (first diff + bad parses)
        deltas = deltas.dropna()

        all_deltas.extend(deltas.tolist())

    # convert to Series for stats
    delta_series = pd.Series(all_deltas)

    print(delta_series.describe())

    # optional diagnostics
    print("\n Negative deltas:", (delta_series < 0).sum())
    print(" Zero deltas:", (delta_series == 0).sum())
    print("\nLarge jumps (>300s):", (delta_series > 300).sum())
    print("\nPercentiles:")
    print(delta_series.quantile([0.5, 0.9, 0.95, 0.99]))

In [18]:
def time_duplicate_ratio_stats(mm_data: pd.DataFrame) -> None:
    traj_duplicate_ratios = []
    traj_zero_delta_ratios = []

    for time_str in mm_data["time_list"].dropna():
        times = pd.to_datetime(time_str.split(","), errors="coerce")

        if len(times) < 2:
            continue

        # duplicate timestamps
        num_duplicates = times.duplicated().sum()
        duplicate_ratio = num_duplicates / len(times)

        # zero deltas
        deltas = times.to_series().diff().dt.total_seconds()
        zero_deltas = (deltas == 0).sum()
        zero_ratio = zero_deltas / (len(times) - 1)

        traj_duplicate_ratios.append(duplicate_ratio)
        traj_zero_delta_ratios.append(zero_ratio)

    dup_series = pd.Series(traj_duplicate_ratios)
    zero_series = pd.Series(traj_zero_delta_ratios)

    print("Duplicate Timestamp Ratio (per trajectory)")
    print(dup_series.describe())

    print("\nZero Delta Ratio (per trajectory)")
    print(zero_series.describe())

    # useful thresholds
    print("\nTrajectories with >10% duplicate timestamps:",
          (dup_series > 0.10).sum())

    print("Trajectories with >10% zero deltas:",
          (zero_series > 0.10).sum())

In [3]:
df_mm_test = pd.read_csv("../baselines/gan/ts_trajgen/data/nyc/nyc_mm_test.csv")
df_mm_train = pd.read_csv("../baselines/gan/ts_trajgen/data/nyc/nyc_mm_train.csv")

df_mm_train.head(3)
# row = df.iloc[0]

# rid = row["rid_list"].split(",")
# time = row["time_list"].split(",")

# print(len(rid), len(time))

,traj_id,rid_list,time_list
0,2340,"112550,112550,112550,112550,112550,112550,1125...","2022-05-13T03:46:31.000,2022-05-13T03:46:33.00..."
1,3678,"112399,112399,112399,112399,112399,112399,1123...","2022-06-19T00:11:49.000,2022-06-19T00:11:52.00..."
2,7152,"111843,111843,111843,111843,111843,111843,1118...","2022-08-19T01:26:39.000,2022-08-19T01:26:40.00..."


In [20]:
df_mm_train["traj_id"].nunique()

8859

In [21]:
df_mm_test["traj_id"].nunique()

2215

If not True then something is off in stitching or trimming maybe in generating the timestamps for each rid script.

In [22]:
check_rid_and_timelist_alignment(df_mm_test) and check_rid_and_timelist_alignment(df_mm_train)

True

Each trajectory should have times that increase only, as trajectories are sequential in nature, thus each rid should its timestamp also increase.

In [23]:
check_monotonic_time(df_mm_test) and check_monotonic_time(df_mm_train)

True

want reasonable spread and not all very short (<5)

In [25]:
check_rid_len_distribution(df_mm_train)

count    8859.000000
mean      296.036121
std       143.054658
min         2.000000
25%       192.000000
50%       263.000000
75%       387.000000
max      1530.000000
Name: rid_list, dtype: float64


In [26]:
check_rid_len_distribution(df_mm_test)

count    2215.000000
mean      299.541309
std       141.894139
min         2.000000
25%       193.500000
50%       266.000000
75%       395.000000
max       724.000000
Name: rid_list, dtype: float64


Are deltas reasonable (not huge spikes)

Are there duplicates (0s): bad interpolation

Are there negatives: broken ordering

Most deltas should be small and smooth

No negatives

Very few zeros

Long tails = expected on highways, but not extreme

**If results look bad…**

correctly align GPS timestamps with FMM output is non-trivial

Because:

tpath introduces hidden edges

im distributing time across edges not directly observed

naive interpolation: uneven or duplicated timestamps

In [27]:
time_list_interpolation_quality(df_mm_test)

count    661269.000000
mean          2.270781
std          13.039172
min           0.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        4189.000000
dtype: float64

 Negative deltas: 0
 Zero deltas: 8

Large jumps (>300s): 151

Percentiles:
0.50     1.0
0.90     3.0
0.95     4.0
0.99    25.0
dtype: float64


In [28]:
time_list_interpolation_quality(df_mm_train)

count    2.613725e+06
mean     2.441600e+00
std      1.779424e+01
min      0.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      6.341000e+03
dtype: float64

 Negative deltas: 0
 Zero deltas: 32

Large jumps (>300s): 740

Percentiles:
0.50     1.0
0.90     3.0
0.95     4.0
0.99    27.0
dtype: float64


**some duplicated timestamps**

Test: 2,196 zeros (~3.7%)

Train: 9,474 zeros (~4.0%)

Zero deltas = some edges share the same timestamp

maybe interpolation is too coarse or very small edges get 0 allocated time

not ideal for generative models

Long tail exists (expected, but worth noting)

Test max: ~5,296 sec (~88 min)

Train max: ~12,286 sec (~3.4 hrs)

Large jumps (>300s):

- Test: 278

- Train: 1,127

These are likely:

long gaps between GPS points

highway segments

missing intermediate observations

maybe because there is a single trajectory for each user and each user is unique, thus
each trajectory is all of a users trajectories combined/concatenated so each trajectory is a "list of trajectories"

In [29]:
time_duplicate_ratio_stats(df_mm_train)

Duplicate Timestamp Ratio (per trajectory)
count    8859.000000
mean        0.000009
std         0.000233
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.016393
dtype: float64

Zero Delta Ratio (per trajectory)
count    8859.000000
mean        0.000009
std         0.000233
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.016423
dtype: float64

Trajectories with >10% duplicate timestamps: 0
Trajectories with >10% zero deltas: 0


In [30]:
time_duplicate_ratio_stats(df_mm_test)

Duplicate Timestamp Ratio (per trajectory)
count    2215.000000
mean        0.000011
std         0.000195
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.004878
dtype: float64

Zero Delta Ratio (per trajectory)
count    2215.000000
mean        0.000011
std         0.000196
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.004902
dtype: float64

Trajectories with >10% duplicate timestamps: 0
Trajectories with >10% zero deltas: 0


**Just filter out bad trajectories???**

cause?:
- Very short time spans + many edges
    - Not enough time to distribute: zeros
- GPS sparsity spikes
    - long jump = many intermediate edges
    - time gets unevenly allocated
- tpath expansion artifacts
    - timestamps not properly redistributed

While over 75% of trajectories exhibit perfect temporal consistency, approximately 5% contain significant duplication artifacts due to interpolation across map-matched paths with uneven edge expansion

## preprocess_pretrain_input.py results

In [4]:
df_pretrain_input_train = pd.read_csv("../baselines/gan/ts_trajgen/data/nyc/nyc_pretrain_input_train.csv")
df_pretrain_input_eval = pd.read_csv("../baselines/gan/ts_trajgen/data/nyc/nyc_pretrain_input_eval.csv")
df_pretrain_input_test = pd.read_csv("../baselines/gan/ts_trajgen/data/nyc/nyc_pretrain_input_test.csv")

In [4]:
df_pretrain_input_train.head()

,trace_loc,trace_time,des,candidate_set,candidate_dis,target
0,3547,1271,126681,"3545,3546","16.91300774265906,17.248733901022256",0
1,30500,1995,27969,"25047,25048,25049","28.893033244781833,28.86003973346032,28.101031...",2
2,111932,266,2921,"114070,114071","21.30771067247634,21.170533131359136",0
3,3693,2589,110219,"2099,2100","29.88686120555259,30.81427285933986",0
4,8079,1064,14097,"8082,8083,8084","3.338628993884467,3.0657894209856282,3.1025362...",2


In [7]:
df_pretrain_input_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 1315 entries, 0 to 1314
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   trace_loc      1315 non-null   str  
 1   trace_time     1315 non-null   str  
 2   des            1315 non-null   int64
 3   candidate_set  1315 non-null   str  
 4   candidate_dis  1315 non-null   str  
 5   target         1315 non-null   int64
dtypes: int64(2), str(4)
memory usage: 166.1 KB


In [8]:
df_pretrain_input_train.describe()

,des,target
count,1315.000000,1315.000000
mean,59239.974144,0.924715
std,49225.027878,0.840176
min,75.000000,0.000000
25%,15807.500000,0.000000
50%,29634.000000,1.000000
75%,112384.000000,1.000000
max,138972.000000,3.000000


In [5]:
df_pretrain_input_eval.head()

,trace_loc,trace_time,des,candidate_set,candidate_dis,target
0,21117,977,23338,"10340,10341,10342","0.790223610997377,1.5842119396345933,1.8620695...",0
1,110249,704,113943,"136170,136171","25.608332658793547,26.15502502526506",0
2,117021,479,112548,"8725,8726,8727","60.80691950119059,59.934700512222754,60.961614...",0
3,"117021,8725","479,480",112548,"115169,115170,115171","60.80691950119059,60.93482969453952,61.2676426...",1
4,"117021,8725,115170","479,480,480",112548,"115157,115158,115159","60.93482969453952,61.0527131066766,60.78341745...",1


In [9]:
df_pretrain_input_eval.info()

<class 'pandas.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   trace_loc      144 non-null    str  
 1   trace_time     144 non-null    str  
 2   des            144 non-null    int64
 3   candidate_set  144 non-null    str  
 4   candidate_dis  144 non-null    str  
 5   target         144 non-null    int64
dtypes: int64(2), str(4)
memory usage: 17.6 KB


In [10]:
df_pretrain_input_eval.describe()

,des,target
count,144.000000,144.000000
mean,60178.152778,0.833333
std,49160.688358,0.747958
min,410.000000,0.000000
25%,15421.250000,0.000000
50%,30741.000000,1.000000
75%,112548.000000,1.000000
max,138956.000000,3.000000


In [6]:
df_pretrain_input_test.head()

,trace_loc,trace_time,des,candidate_set,candidate_dis,target
0,129080,880,98211,"101986,101987,101988","22.13352571937194,21.70034649254864,21.5126177...",2
1,118205,49,6469,"135612,135613,135614","16.329780359879592,17.10500065368803,16.410185...",2
2,"126738,3407","156,169",138813,"3408,3409","36.53565797526367,36.96738546045091",0
3,1427,124,5555,"1430,1431,1432","4.991402647334172,4.887785581022797,5.37227953...",1
4,18700,453,30459,"18701,18702,18703,18704","23.128341634254923,23.87393030251104,22.880315...",2


In [11]:
df_pretrain_input_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 326 entries, 0 to 325
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   trace_loc      326 non-null    str  
 1   trace_time     326 non-null    str  
 2   des            326 non-null    int64
 3   candidate_set  326 non-null    str  
 4   candidate_dis  326 non-null    str  
 5   target         326 non-null    int64
dtypes: int64(2), str(4)
memory usage: 40.0 KB


In [12]:
df_pretrain_input_test.describe()

,des,target
count,326.000000,326.000000
mean,60786.733129,0.972393
std,49392.030205,0.864250
min,464.000000,0.000000
25%,18046.000000,0.000000
50%,30492.000000,1.000000
75%,112531.000000,2.000000
max,138813.000000,3.000000


#### checking for NaN/inf in candidate_dis

In [6]:
for pretrain_input_df in [df_pretrain_input_train, df_pretrain_input_eval, df_pretrain_input_test]:
    vals = []
    for s in pretrain_input_df["candidate_dis"]:
        vals.extend(float(x) for x in str(s).split(",") if x)

    arr = np.array(vals)
    print(np.isnan(arr).sum(), np.isinf(arr).sum(), arr.min(), arr.max())
    # print(pretrain_input_df, np.isnan(arr).sum(), np.isinf(arr).sum(), arr.min(), arr.max())

0 0 0.0 193.45224424101357
0 0 0.0 126.71655758930841
0 0 0.0 112.1603655270884


## pretrain_gat_fc.py results

#### checking node_feature.pt for NaN/inf

pretrain_gat_fc.py runs with NYC symlinked as Xian, but GAT loss becomes NaN, likely due to invalid node features or distance inputs; accuracy is not reliable until NaN source is fixed.

In [11]:
x = torch.load("../baselines/gan/ts_trajgen/repo/data/Xian/node_feature.pt")
print(x.shape)
print(torch.isnan(x).sum(), torch.isinf(x).sum())
print(x.min(), x.max())

torch.Size([139155, 12])
tensor(0) tensor(0)
tensor(0.) tensor(110.)
